1. Cài đặt datasets phiên bản 2.20.0 để tương thích với bộ dữ liệu cũ (Khi dùng trên web)

In [ ]:
!pip install transformers datasets pyvi pandas scikit-learn accelerate -U
!pip install datasets==2.20.0
!pip install transformers pyvi pandas scikit-learn accelerate -U

2. Khai báo thư viện và cấu hình   

In [ ]:
import pandas as pd
import torch
import numpy as np
import os
import shutil
from datasets import load_dataset, Dataset
from pyvi import ViTokenizer
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments
)
import unicodedata
import re


MODEL_NAME = "vinai/phobert-base"
MAX_LEN = 128
EPOCHS = 3
BATCH_SIZE = 16

print(f"✅ Đã import thư viện. Sử dụng Model: {MODEL_NAME}")

3. Xây dựng bộ lọc dữ liệu (Cleaning Tools)

In [ ]:
teencode_dict = {"ko": "không", "k": "không", "dc": "được", "t": "tôi", "mk": "mình", "wá": "quá"}

stopwords = set([
    "thì", "là", "mà", "và", "của", "những", "cái", "việc",
    "ở", "với", "cho", "được", "bị", "các", "có", "trong",
    "đã", "đang", "sẽ", "cũng", "này", "kia", "đó", "ra", "vào",
    "mình", "tôi", "bạn", "nó", "họ", "rất", "quá", "lắm"
])

def clean_text(text):
    text = str(text).lower()
    text = unicodedata.normalize('NFC', text)

    words = text.split()
    words = [teencode_dict.get(w, w) for w in words]
    text = ' '.join(words)
    text = ViTokenizer.tokenize(text)
    tokens = text.split()
    filtered_tokens = [t for t in tokens if t not in stopwords]

    return ' '.join(filtered_tokens)

print("✅ Đã thiết lập xong hàm xử lý dữ liệu (Clean Text).")

4. Tải và Áp dụng làm sạch dữ liệu

In [ ]:
print("🚀 ĐANG TẢI VÀ XỬ LÝ DỮ LIỆU...")

dataset = load_dataset("uitnlp/vietnamese_students_feedback", trust_remote_code=True)
train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

print("⏳ Đang làm sạch và loại bỏ từ vô nghĩa...")
train_df['text'] = train_df['sentence'].apply(clean_text)
test_df['text'] = test_df['sentence'].apply(clean_text)

train_df['label'] = train_df['sentiment']
test_df['label'] = test_df['sentiment']

train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['text', 'label']])

print(f"✅ Xử lý xong. Train size: {len(train_dataset)}")
print(f"🔍 Mẫu dữ liệu sau khi clean: {train_dataset[0]['text']}")

5. Tokenization (Mã hóa cho PhoBERT)

In [ ]:
print("⚙️ Đang tải Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=MAX_LEN)

print("⏳ Đang mã hóa dữ liệu...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)
print("✅ Tokenization hoàn tất.")

6. Huấn luyện mô hình

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {'accuracy': acc, 'f1': f1}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    fp16=True, # Bật nếu dùng GPU T4 trên Colab để train nhanh hơn
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("\n🏋️ BẮT ĐẦU TRAIN (Method 2: Clean Data)...")
trainer.train()

print("\n📊 ĐÁNH GIÁ MODEL...")
eval_result = trainer.evaluate()
print(f"🎯 Accuracy: {eval_result['eval_accuracy']:.4f}")

7. Lưu và trích xuất mô hình

In [ ]:
print("\n💾 ĐANG LƯU MODEL VÀO FOLDER 'model_save'...")
output_dir = "model_save"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("✅ Đã lưu xong model!")

shutil.make_archive('my_sentiment_model_v2', 'zip', 'model_save')
from google.colab import files
files.download('my_sentiment_model_v2.zip')